In [1]:
import torch
from torch import nn
import torch.nn.functional as F

### Exercise 2

Assume that we want to reuse only parts of a network to be incorporated into a network having a different architecture. How would you go about using, say the first two layers from a previous network in a new network?

In [2]:
class OldMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64),   
            nn.ReLU(),
            nn.Linear(64, 32),   
            nn.ReLU(),
            nn.Linear(32, 10)    
        )

    def forward(self, x):
        return self.net(x)

In [3]:
old_net = OldMLP()

In [4]:
X = torch.randn(4, 20)
old_out = old_net(X)
print(old_out.shape)

torch.Size([4, 10])


In [5]:
feature_extractor = nn.Sequential(*list(old_net.net.children())[:4])
print(feature_extractor)

Sequential(
  (0): Linear(in_features=20, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
)


In [6]:
class NewMLP(nn.Module):
    def __init__(self, reused_layers):
        super().__init__()
        self.features = reused_layers
        self.new_head = nn.Linear(32, 5)

    def forward(self, x):
        x = self.features(x)
        x = self.new_head(x)
        return x

In [7]:
new_net = NewMLP(feature_extractor)

In [8]:
new_out = new_net(X)
print(new_out.shape)

torch.Size([4, 5])


In [9]:
print(old_net.net[0].weight is new_net.features[0].weight)
print(old_net.net[2].weight is new_net.features[2].weight)

True
True
